# DSI-CL PoC: T5Gemma 2 + LoRA with Chrono-Semantic DocIDs

**Goal**: Verify that T5Gemma 2 can do basic DocID generation — train on query→DocID pairs, then retrieve documents via constrained beam search.

**What this notebook does**:
1. Load 334 CC-News articles (621 chunks, 1860 queries in 3 styles) from our generated dataset
2. Explore T5Gemma 2 architecture
3. Create document embeddings using the T5Gemma encoder
4. Train FAISS RQ codebook → build chrono-semantic DocIDs
5. Extend tokenizer with special tokens (year, month, RQ)
6. Fine-tune with LoRA on query→DocID (multiple queries per document)
7. Evaluate: constrained beam search → retrieve documents

**Runtime**: A100 GPU on Colab (~20-30 min total)

## 0. Setup

In [ ]:
# ============================================================
# 0A. Clone repo + install dependencies
# ============================================================
import os

REPO_URL = "https://github.com/YuvalShemla/DSI-news.git"
REPO_DIR = "/content/DSI-CL"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f"Cloned repo to {REPO_DIR}")
else:
    print(f"Repo already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ============================================================
# 0B. Install dependencies
# ============================================================
# NOTE: transformers must be >=4.51 for T5Gemma support
!pip install -q transformers>=4.51.0 peft>=0.15.0 faiss-cpu>=1.8.0 \
    accelerate pytrec_eval pyyaml ujson sentencepiece \
    matplotlib seaborn plotly 2>&1 | tail -3

# Verify key versions
import transformers, peft, torch, faiss
print(f"transformers: {transformers.__version__}")
print(f"peft:          {peft.__version__}")
print(f"torch:         {torch.__version__}")
print(f"faiss:         {faiss.__version__}")
print(f"CUDA:          {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ============================================================
# 0C. HuggingFace login (required for T5Gemma — Gemma license)
# ============================================================
# You must:
#   1. Go to https://huggingface.co/google/t5gemma-2-270m-270m
#   2. Accept the Gemma license
#   3. Create a token at https://huggingface.co/settings/tokens
#   4. Paste it below or set as Colab secret

from huggingface_hub import login
from google.colab import userdata

try:
    # Try Colab secrets first
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Logged in via Colab secret")
except Exception:
    print("Set HF_TOKEN in Colab secrets, or run: huggingface-cli login")
    login()

In [ ]:
# ============================================================
# 0D. Global config
# ============================================================
import json
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('poc')

# --- CONFIG ---
MODEL_NAME = "google/t5gemma-2-270m-270m"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16

# DocID params
NUM_RQ_CODEBOOKS = 4   # 4 codebooks for PoC (6 for full scale)
RQ_NBITS = 5           # 32 centroids per codebook for PoC (8→256 for full scale)
RQ_CODEBOOK_SIZE = 2 ** RQ_NBITS  # 32
YEAR_RANGE = (2017, 2018)  # CC-News date range
DOCID_LENGTH = 2 + NUM_RQ_CODEBOOKS  # 2 chrono + N RQ = 6 tokens

# Training params
LORA_R = 8
LORA_ALPHA = 16
LEARNING_RATE = 5e-4
NUM_EPOCHS = 15         # more data now (1860 pairs), fewer epochs needed
BATCH_SIZE = 16         # larger batch — more data
MAX_QUERY_LEN = 96      # longer queries in v3 (factual questions)

# Eval params
NUM_BEAMS = 50

DATA_PATH = Path("data/queries/poc_data.csv")

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"DocID length: {DOCID_LENGTH} tokens (2 chrono + {NUM_RQ_CODEBOOKS} RQ)")
print(f"RQ codebook: {NUM_RQ_CODEBOOKS} codebooks x {RQ_CODEBOOK_SIZE} centroids")

---
## 1. Load & Explore PoC Data

In [ ]:
# ============================================================
# 1A. Load the PoC dataset
# ============================================================
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} query-document pairs")
print(f"Columns: {list(df.columns)}")
print(f"Unique documents (chunk_id): {df['chunk_id'].nunique()}")
print(f"Unique articles (doc_id):    {df['doc_id'].nunique()}")
print(f"Query versions: {df['query_version'].value_counts().to_dict()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

# Build a deduplicated docs dataframe (one row per chunk)
docs_df = df.drop_duplicates(subset='chunk_id')[['chunk_id', 'doc_id', 'date', 'chunk_text']].reset_index(drop=True)
print(f"\nDeduplicated docs: {len(docs_df)} chunks from {docs_df['doc_id'].nunique()} articles")
docs_df.head()

In [ ]:
# ============================================================
# 1B. Data distribution visualizations
# ============================================================
docs_df['date_parsed'] = pd.to_datetime(docs_df['date'])
docs_df['year'] = docs_df['date_parsed'].dt.year
docs_df['month'] = docs_df['date_parsed'].dt.month
docs_df['year_month'] = docs_df['date_parsed'].dt.to_period('M').astype(str)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Chunks per year
docs_df['year'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Chunks per Year')
axes[0,0].set_ylabel('Count')

# Chunks per month
docs_df['year_month'].value_counts().sort_index().plot(kind='bar', ax=axes[0,1], color='coral')
axes[0,1].set_title('Chunks per Year-Month')
axes[0,1].tick_params(axis='x', rotation=45)

# Text length distribution
docs_df['text_len'] = docs_df['chunk_text'].str.split().str.len()
docs_df['text_len'].hist(bins=30, ax=axes[1,0], color='seagreen')
axes[1,0].set_title('Chunk Length (words)')
axes[1,0].set_xlabel('Words')

# Query length by version
for version in df['query_version'].unique():
    subset = df[df['query_version'] == version]
    qlen = subset['query'].str.split().str.len()
    axes[1,1].hist(qlen, bins=20, alpha=0.5, label=version)
axes[1,1].set_title('Query Length by Version (words)')
axes[1,1].set_xlabel('Words')
axes[1,1].legend()

plt.tight_layout()
plt.show()

print(f"Avg chunk length:  {docs_df['text_len'].mean():.0f} words")
print(f"Avg query length:  {df['query'].str.split().str.len().mean():.1f} words")
print(f"Queries per chunk: {df.groupby('chunk_id').size().mean():.1f}")

---
## 2. Load & Explore T5Gemma 2 Architecture

In [ ]:
# ============================================================
# 2A. Load the base model
# ============================================================
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print(f"Loading {MODEL_NAME}...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"\n{'='*60}")
print(f"  T5Gemma 2 Architecture Summary")
print(f"{'='*60}")
print(f"  Model type:       {base_model.config.model_type}")
print(f"  Vocab size:       {base_model.config.vocab_size:,}")
print(f"  Tie embeddings:   {base_model.config.tie_word_embeddings}")
print(f"  Total params:     {sum(p.numel() for p in base_model.parameters()):,}")
print(f"  Embedding shape:  {base_model.get_input_embeddings().weight.shape}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# 2B. Inspect encoder and decoder configs
# ============================================================
enc_cfg = base_model.config.encoder
dec_cfg = base_model.config.decoder

print(f"Encoder config type: {type(enc_cfg).__name__}")
print(f"Decoder config type: {type(dec_cfg).__name__}")

# Dump all attributes to discover the actual field names
print("\nENCODER config (all attributes):")
for k in sorted(vars(enc_cfg)):
    if not k.startswith('_'):
        print(f"  {k:35s} = {getattr(enc_cfg, k)}")

print(f"\nDECODER config (key attributes):")
for k in ['hidden_size', 'intermediate_size', 'num_hidden_layers', 'num_attention_heads',
          'num_key_value_heads', 'head_dim', 'max_position_embeddings']:
    val = getattr(dec_cfg, k, 'N/A')
    print(f"  {k:30s} = {val}")

# Get encoder hidden dim — try multiple possible attribute names
ENCODER_DIM = None
for attr in ['hidden_size', 'd_model', 'embedding_dim', 'dim']:
    if hasattr(enc_cfg, attr):
        ENCODER_DIM = getattr(enc_cfg, attr)
        print(f"\n-> Encoder hidden dim: {ENCODER_DIM} (from enc_cfg.{attr})")
        break

if ENCODER_DIM is None:
    # Fallback: get it directly from the embedding layer
    ENCODER_DIM = base_model.get_input_embeddings().weight.shape[1]
    print(f"\n-> Encoder hidden dim: {ENCODER_DIM} (from embedding weight shape)")

print(f"-> Decoder hidden dim: {dec_cfg.hidden_size}")

In [ ]:
# ============================================================
# 2C. Find LoRA target modules — inspect named modules
# ============================================================
# We need to find the exact projection layer names for LoRA
print("All unique module types in the model:")
module_types = set()
linear_names = set()
for name, module in base_model.named_modules():
    module_types.add(type(module).__name__)
    if isinstance(module, torch.nn.Linear):
        # Extract the leaf name (e.g., 'q_proj' from 'encoder.layers.0.self_attn.q_proj')
        leaf = name.split('.')[-1]
        linear_names.add(leaf)

print(f"\nModule types: {sorted(module_types)}")
print(f"\nLinear layer names (candidates for LoRA):")
for name in sorted(linear_names):
    print(f"  - {name}")

# Auto-detect attention projection layers
# IMPORTANT: exclude 'out_proj' — that's the lm_head output projection, not attention.
# With tie_word_embeddings + vocab resize, LoRA on out_proj causes dimension mismatch.
ATTN_PROJ_NAMES = [n for n in linear_names if n in ('q_proj', 'k_proj', 'v_proj', 'o_proj')]
if not ATTN_PROJ_NAMES:
    # Fallback: use proj layers but explicitly exclude out_proj
    ATTN_PROJ_NAMES = [n for n in linear_names if 'proj' in n and n != 'out_proj']
print(f"\n-> Selected LoRA targets: {sorted(ATTN_PROJ_NAMES)}")
print(f"   (excluded 'out_proj' = lm_head, would conflict with tied embeddings + vocab resize)")

In [ ]:
# ============================================================
# 2D. Visualize parameter distribution by component
# ============================================================
param_counts = {'encoder': 0, 'decoder': 0, 'embeddings': 0, 'lm_head': 0, 'other': 0}
for name, p in base_model.named_parameters():
    n = p.numel()
    if 'encoder' in name and 'embed' not in name:
        param_counts['encoder'] += n
    elif 'decoder' in name and 'embed' not in name:
        param_counts['decoder'] += n
    elif 'embed' in name:
        param_counts['embeddings'] += n
    elif 'lm_head' in name:
        param_counts['lm_head'] += n
    else:
        param_counts['other'] += n

fig, ax = plt.subplots(figsize=(8, 4))
components = list(param_counts.keys())
counts = [param_counts[c] / 1e6 for c in components]
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
bars = ax.bar(components, counts, color=colors)
ax.set_ylabel('Parameters (millions)')
ax.set_title('T5Gemma 2 Parameter Distribution')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{count:.1f}M', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

---
## 3. Compute Document Embeddings (T5Gemma Encoder)

In [ ]:
# ============================================================
# 3A. Encode documents using T5Gemma's encoder
# ============================================================
# We use the encoder to get dense representations for RQ codebook training.
# For each chunk, we encode the text and take mean pooling.

base_model.to(DEVICE)
base_model.eval()

# Use deduplicated docs (one embedding per chunk)
doc_texts = docs_df['chunk_text'].tolist()

# Truncate very long texts for embedding (keep first ~300 words)
doc_texts = [' '.join(t.split()[:300]) for t in doc_texts]

print(f"Encoding {len(doc_texts)} document chunks...")
print(f"DEBUG: Sample input: {doc_texts[0][:120]}...")

embeddings = []
EMBED_BATCH = 8

with torch.no_grad():
    for i in range(0, len(doc_texts), EMBED_BATCH):
        batch_texts = doc_texts[i:i+EMBED_BATCH]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors='pt'
        ).to(DEVICE)

        # Get encoder outputs
        encoder_out = base_model.get_encoder()(**inputs)
        # Mean pooling over non-padding tokens
        mask = inputs['attention_mask'].unsqueeze(-1).float()
        pooled = (encoder_out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
        embeddings.append(pooled.cpu().float().numpy())

        if i % 40 == 0:
            print(f"  Batch {i//EMBED_BATCH + 1}/{(len(doc_texts)-1)//EMBED_BATCH + 1}")

doc_embeddings = np.vstack(embeddings)
print(f"\nEmbeddings shape: {doc_embeddings.shape}")
print(f"Embedding dim: {doc_embeddings.shape[1]}")
print(f"Embedding norm (mean): {np.linalg.norm(doc_embeddings, axis=1).mean():.2f}")

In [ ]:
# ============================================================
# 3B. Visualize embeddings with t-SNE
# ============================================================
from sklearn.manifold import TSNE

# t-SNE reduction
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(doc_embeddings)-1))
emb_2d = tsne.fit_transform(doc_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Color by year
years = docs_df['year'].values
for yr in sorted(set(years)):
    mask = years == yr
    axes[0].scatter(emb_2d[mask, 0], emb_2d[mask, 1], s=8, alpha=0.6, label=str(yr))
axes[0].set_title(f't-SNE of {len(doc_embeddings)} Document Embeddings (by Year)')
axes[0].legend()

# Color by year-month
year_months = docs_df['year_month'].values
unique_ym = sorted(set(year_months))
cmap = plt.cm.get_cmap('tab20', len(unique_ym))
for idx, ym in enumerate(unique_ym):
    mask = year_months == ym
    axes[1].scatter(emb_2d[mask, 0], emb_2d[mask, 1], s=8, alpha=0.6, color=cmap(idx), label=ym)
axes[1].set_title('t-SNE by Year-Month')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

---
## 4. Train FAISS RQ & Build Chrono-Semantic DocIDs

In [ ]:
# ============================================================
# 4A. Train FAISS Residual Quantizer
# ============================================================
import faiss
from sklearn.decomposition import PCA

n_docs, embed_dim_orig = doc_embeddings.shape

# FAISS RQ uses internal PCA for beam search, which fails when n_docs <= embed_dim.
# Reduce dimensionality first — this is standard practice for RQ on high-dim embeddings.
RQ_DIM = min(256, n_docs - 1, embed_dim_orig)
if n_docs <= embed_dim_orig or embed_dim_orig > 256:
    print(f"PCA reducing embeddings: {embed_dim_orig} -> {RQ_DIM} (n_docs={n_docs} <= embed_dim or dim>256)")
    pca = PCA(n_components=RQ_DIM)
    doc_embeddings_rq = pca.fit_transform(doc_embeddings).astype(np.float32)
    print(f"  PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")
else:
    doc_embeddings_rq = doc_embeddings.astype(np.float32)
    RQ_DIM = embed_dim_orig

embed_dim = RQ_DIM  # This is the dim used for RQ centroids
print(f"Training RQ: {n_docs} docs, {embed_dim} dims, {NUM_RQ_CODEBOOKS} codebooks, {RQ_NBITS} bits")

rq_index = faiss.IndexResidualQuantizer(embed_dim, NUM_RQ_CODEBOOKS, RQ_NBITS)
rq_index.train(doc_embeddings_rq)
print("RQ training complete!")

# Extract RQ codes for all documents
rq = rq_index.rq
unit8_codes = rq.compute_codes(doc_embeddings_rq)

doc_rq_codes = []
for u8_code in unit8_codes:
    bs = faiss.BitstringReader(faiss.swig_ptr(u8_code), unit8_codes.shape[1])
    code = [bs.read(RQ_NBITS) for _ in range(NUM_RQ_CODEBOOKS)]
    doc_rq_codes.append(code)

print(f"\nRQ codes shape: ({len(doc_rq_codes)}, {len(doc_rq_codes[0])})")
print(f"Sample RQ codes (first 5 docs):")
for i in range(min(5, len(doc_rq_codes))):
    print(f"  {docs_df.iloc[i]['chunk_id'][:50]}...: {doc_rq_codes[i]}")

In [ ]:
# ============================================================
# 4B. Build chrono-semantic DocIDs
# ============================================================
# DocID = [year_code, month_code, rq_0, rq_1, ..., rq_{N-1}]

def build_chrono_docid(date_str, rq_codes, year_range=YEAR_RANGE):
    """Build a chrono-semantic DocID from date + RQ codes."""
    date = pd.to_datetime(date_str)
    year_code = date.year - year_range[0]  # 0-indexed
    month_code = date.month - 1            # 0-indexed
    return [year_code, month_code] + rq_codes

# Build DocIDs for all document chunks
docid_to_smtid = {}  # chunk_id -> [year_code, month_code, rq0, ..., rqN]
for i, row in docs_df.iterrows():
    chunk_id = row['chunk_id']
    smtid = build_chrono_docid(row['date'], doc_rq_codes[i])
    docid_to_smtid[chunk_id] = smtid

print(f"Built {len(docid_to_smtid)} chrono-semantic DocIDs")
print(f"DocID length: {len(list(docid_to_smtid.values())[0])} tokens")
print(f"\nSample DocIDs:")
for chunk_id, smtid in list(docid_to_smtid.items())[:5]:
    date = docs_df[docs_df['chunk_id'] == chunk_id]['date'].values[0]
    print(f"  {chunk_id[:50]}... ({date}): year={smtid[0]}, month={smtid[1]}, rq={smtid[2:]}")

In [ ]:
# ============================================================
# 4C. Generate special tokens + extend tokenizer
# ============================================================

def generate_special_tokens(year_range, num_codebooks, codebook_size):
    """Generate special token strings in canonical order."""
    tokens = []
    for y in range(year_range[0], year_range[1] + 1):
        tokens.append(f"<year_{y}>")
    for m in range(1, 13):
        tokens.append(f"<month_{m:02d}>")
    for cb in range(num_codebooks):
        for code in range(codebook_size):
            tokens.append(f"<rq_{cb}_{code}>")
    return tokens

special_tokens = generate_special_tokens(YEAR_RANGE, NUM_RQ_CODEBOOKS, RQ_CODEBOOK_SIZE)
n_year = YEAR_RANGE[1] - YEAR_RANGE[0] + 1
n_month = 12
n_rq = NUM_RQ_CODEBOOKS * RQ_CODEBOOK_SIZE

print(f"Special tokens breakdown:")
print(f"  Year tokens:  {n_year} ({special_tokens[0]} .. {special_tokens[n_year-1]})")
print(f"  Month tokens: {n_month} ({special_tokens[n_year]} .. {special_tokens[n_year+n_month-1]})")
print(f"  RQ tokens:    {n_rq} ({special_tokens[n_year+n_month]} .. {special_tokens[-1]})")
print(f"  Total:        {len(special_tokens)}")

# Add to tokenizer
base_vocab_size = len(tokenizer)
num_added = tokenizer.add_tokens(special_tokens)
base_model.resize_token_embeddings(len(tokenizer))

print(f"\nBase vocab: {base_vocab_size:,} → Extended vocab: {len(tokenizer):,} (+{num_added})")
print(f"Embedding shape after resize: {base_model.get_input_embeddings().weight.shape}")

In [ ]:
# ============================================================
# 4D. Initialize chrono embeddings (mean + noise)
# ============================================================
embed_weight = base_model.get_input_embeddings().weight
mean_embed = embed_weight.data[:base_vocab_size].mean(dim=0)

with torch.no_grad():
    for y in range(YEAR_RANGE[0], YEAR_RANGE[1] + 1):
        tid = tokenizer.convert_tokens_to_ids(f"<year_{y}>")
        embed_weight.data[tid] = mean_embed + torch.randn_like(mean_embed) * 0.01
    for m in range(1, 13):
        tid = tokenizer.convert_tokens_to_ids(f"<month_{m:02d}>")
        embed_weight.data[tid] = mean_embed + torch.randn_like(mean_embed) * 0.01

print("Initialized year/month embeddings from mean + noise")

# Initialize RQ embeddings from centroids
rq_centroids = faiss.vector_to_array(rq.codebooks).reshape(NUM_RQ_CODEBOOKS, RQ_CODEBOOK_SIZE, embed_dim)
model_dim = embed_weight.shape[1]

# Project centroids if dimensions differ
if embed_dim != model_dim:
    print(f"Projecting RQ centroids: {embed_dim} → {model_dim}")
    proj = torch.randn(embed_dim, model_dim, dtype=torch.float32) * (1.0 / embed_dim ** 0.5)
    centroids_proj = torch.from_numpy(rq_centroids).float() @ proj
else:
    centroids_proj = torch.from_numpy(rq_centroids).float()

with torch.no_grad():
    for cb in range(NUM_RQ_CODEBOOKS):
        for code in range(RQ_CODEBOOK_SIZE):
            tid = tokenizer.convert_tokens_to_ids(f"<rq_{cb}_{code}>")
            embed_weight.data[tid] = centroids_proj[cb, code].to(embed_weight.dtype)

print(f"Initialized {NUM_RQ_CODEBOOKS * RQ_CODEBOOK_SIZE} RQ embeddings from centroids")

In [ ]:
# ============================================================
# 4E. Build docid_to_tokenids mapping
# ============================================================
# Convert raw codes to absolute token IDs

docid_to_tokenids = {}
for chunk_id, smtid in docid_to_smtid.items():
    year_code, month_code = smtid[0], smtid[1]
    rq_codes = smtid[2:]

    year_token = f"<year_{YEAR_RANGE[0] + year_code}>"
    month_token = f"<month_{month_code + 1:02d}>"
    rq_tokens = [f"<rq_{cb}_{code}>" for cb, code in enumerate(rq_codes)]

    token_ids = [
        tokenizer.convert_tokens_to_ids(year_token),
        tokenizer.convert_tokens_to_ids(month_token),
    ] + [tokenizer.convert_tokens_to_ids(t) for t in rq_tokens]

    docid_to_tokenids[chunk_id] = token_ids

# Verify
print(f"Built docid_to_tokenids for {len(docid_to_tokenids)} chunks")
print(f"\nSample mappings:")
for chunk_id in list(docid_to_tokenids.keys())[:3]:
    tids = docid_to_tokenids[chunk_id]
    tokens = [tokenizer.convert_ids_to_tokens(t) for t in tids]
    print(f"  {chunk_id[:50]}...")
    print(f"    ids={tids}")
    print(f"    tokens={tokens}")

# Check uniqueness
smtid_strs = ['_'.join(str(t) for t in tids) for tids in docid_to_tokenids.values()]
n_unique = len(set(smtid_strs))
print(f"\nUnique DocID sequences: {n_unique}/{len(smtid_strs)} ({100*n_unique/len(smtid_strs):.1f}%)")
if n_unique < len(smtid_strs):
    print(f"  WARNING: {len(smtid_strs) - n_unique} collisions — some chunks share a DocID")

---
## 5. Build Prefix Trie & Prepare Training Data

In [ ]:
# ============================================================
# 5A. Build prefix trie for constrained beam search
# ============================================================
from collections import defaultdict

# Build the trie in-memory (no need for files in PoC)
prefix_dict = defaultdict(set)
for chunk_id, tokenids in docid_to_tokenids.items():
    extended = [tokenizer.pad_token_id] + tokenids
    for i in range(1, len(extended)):
        prefix_dict[tuple(extended[:i])].add(extended[i])

# Stats
from collections import Counter
prefix_len_counts = Counter(len(k) for k in prefix_dict)
print("Prefix trie statistics:")
print(f"  Total entries: {len(prefix_dict)}")
for plen in sorted(prefix_len_counts):
    print(f"  Prefix length {plen}: {prefix_len_counts[plen]} entries, avg fan-out {sum(len(v) for k,v in prefix_dict.items() if len(k)==plen)/prefix_len_counts[plen]:.1f}")

# The prefixer function for model.generate()
def prefixer_fn(batch_id, sent):
    """Constrained decoding: return allowed next tokens for given prefix."""
    key = tuple(sent.cpu().tolist())
    return list(prefix_dict.get(key, set()))

# Also build reverse map: token_id string -> chunk_ids
smtid_to_docids = defaultdict(list)
for chunk_id, tokenids in docid_to_tokenids.items():
    key = '_'.join(str(t) for t in tokenids)
    smtid_to_docids[key].append(chunk_id)

print(f"\nReverse map: {len(smtid_to_docids)} unique sequences -> {len(docid_to_tokenids)} chunks")
n_collisions = sum(1 for v in smtid_to_docids.values() if len(v) > 1)
if n_collisions:
    print(f"  {n_collisions} sequences map to multiple chunks (collisions)")

In [ ]:
# ============================================================
# 5B. Create training dataset: (query, docid_token_ids) pairs
# ============================================================
# Each chunk has ~3 queries (v1_keyword, v2_topical, v3_factual) = more training signal
from torch.utils.data import Dataset, DataLoader

class DocIDDataset(Dataset):
    """Dataset: (query_text, docid_token_ids) with multiple queries per document."""
    def __init__(self, query_df, docid_to_tokenids):
        self.examples = []
        skipped = 0
        for _, row in query_df.iterrows():
            chunk_id = row['chunk_id']
            query = row['query']
            if chunk_id in docid_to_tokenids:
                self.examples.append((query, docid_to_tokenids[chunk_id], chunk_id))
            else:
                skipped += 1
        print(f"DEBUG: Created dataset with {len(self.examples)} examples ({skipped} skipped)")
        print(f"  Unique chunks referenced: {len(set(ex[2] for ex in self.examples))}")
        print(f"  Avg queries per chunk: {len(self.examples)/len(set(ex[2] for ex in self.examples)):.1f}")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        query, tokenids, chunk_id = self.examples[idx]
        return query, tokenids


class DocIDCollator:
    """Tokenize queries and build decoder inputs."""
    def __init__(self, tokenizer, max_length=96):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        queries, docid_lists = zip(*batch)
        tok = self.tokenizer(
            list(queries), padding='longest', truncation=True,
            max_length=self.max_length, return_tensors='pt'
        )
        labels = torch.LongTensor(list(docid_lists))
        bsz = labels.size(0)
        prefix = torch.full((bsz, 1), self.tokenizer.pad_token_id, dtype=torch.long)
        decoder_input_ids = torch.cat([prefix, labels[:, :-1]], dim=1)

        return {
            'input_ids': tok['input_ids'],
            'attention_mask': tok['attention_mask'],
            'decoder_input_ids': decoder_input_ids,
            'labels': labels,
        }


train_dataset = DocIDDataset(df, docid_to_tokenids)
collator = DocIDCollator(tokenizer, max_length=MAX_QUERY_LEN)

# Quick sanity check
sample_batch = collator([train_dataset[0], train_dataset[1]])
print(f"\nSanity check batch:")
for k, v in sample_batch.items():
    print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
print(f"  labels[0]: {sample_batch['labels'][0].tolist()}")
print(f"  decoded:   {[tokenizer.convert_ids_to_tokens(t) for t in sample_batch['labels'][0].tolist()]}")

---
## 6. Apply LoRA & Train

In [ ]:
# ============================================================
# 6A. Apply LoRA
# ============================================================
from peft import LoraConfig, TaskType, get_peft_model

# Use the target modules we discovered in step 2C
lora_targets = sorted(ATTN_PROJ_NAMES) if ATTN_PROJ_NAMES else ["q_proj", "v_proj"]
print(f"Applying LoRA to: {lora_targets}")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=lora_targets,
    lora_dropout=0.05,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal: {total_params:,}  Trainable: {trainable_params:,}  ({100*trainable_params/total_params:.2f}%)")

In [ ]:
# ============================================================
# 6B. Training loop
# ============================================================
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collator, num_workers=0
)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# Training
loss_history = []
model.train()

print(f"Training for {NUM_EPOCHS} epochs, {len(train_loader)} batches/epoch, {len(train_dataset)} examples")
print(f"{'Epoch':>5} {'Loss':>10} {'LR':>12}")
print("-" * 30)

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    n_batches = 0

    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            decoder_input_ids=batch['decoder_input_ids'],
            labels=batch['labels'],
        )
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    scheduler.step()
    avg_loss = epoch_loss / n_batches
    loss_history.append(avg_loss)
    lr = scheduler.get_last_lr()[0]

    if epoch % 3 == 0 or epoch == NUM_EPOCHS - 1:
        print(f"{epoch+1:5d} {avg_loss:10.4f} {lr:12.6f}")

print(f"\nTraining complete! Final loss: {loss_history[-1]:.4f}")

In [ ]:
# ============================================================
# 6C. Visualize training loss
# ============================================================
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(loss_history)+1), loss_history, 'b-o', markersize=3, linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss (Query → DocID)')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"Loss: {loss_history[0]:.4f} (start) → {loss_history[-1]:.4f} (end)")
print(f"Reduction: {(1 - loss_history[-1]/loss_history[0])*100:.1f}%")

---
## 7. Evaluate: Constrained Beam Search Retrieval

In [ ]:
# ============================================================
# 7A. Run constrained beam search on all queries
# ============================================================
from transformers import GenerationConfig

model.eval()
generation_config = GenerationConfig.from_model_config(model.config)

qid_to_rankdata = {}  # qid -> {chunk_id: score}
qid_to_query = {}     # for display
qid_to_chunk = {}     # ground truth chunk_id

eval_loader = DataLoader(
    train_dataset, batch_size=8, shuffle=False,
    collate_fn=collator, num_workers=0
)

print(f"Running constrained beam search (beams={NUM_BEAMS})...")
all_queries = [ex[0] for ex in train_dataset.examples]
all_chunks = [ex[2] for ex in train_dataset.examples]
query_idx = 0

for batch_i, batch in enumerate(eval_loader):
    bsz = batch['input_ids'].size(0)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            generation_config=generation_config,
            prefix_allowed_tokens_fn=prefixer_fn,
            max_new_tokens=DOCID_LENGTH,
            num_beams=NUM_BEAMS,
            num_return_sequences=min(NUM_BEAMS, len(docid_to_tokenids)),
            output_scores=True,
            return_dict_in_generate=True,
        )

    n_return = min(NUM_BEAMS, len(docid_to_tokenids))
    seqs = outputs.sequences[:, 1:]  # remove decoder start token
    scores = outputs.sequences_scores

    for i in range(bsz):
        qid = str(query_idx)
        qid_to_query[qid] = all_queries[query_idx]
        qid_to_chunk[qid] = all_chunks[query_idx]
        qid_to_rankdata[qid] = {}

        for j in range(n_return):
            seq_idx = i * n_return + j
            if seq_idx >= len(seqs):
                break
            token_ids = seqs[seq_idx].cpu().tolist()[:DOCID_LENGTH]
            smtid_key = '_'.join(str(t) for t in token_ids)
            score = scores[seq_idx].item()

            if smtid_key in smtid_to_docids:
                for chunk_id in smtid_to_docids[smtid_key]:
                    if chunk_id not in qid_to_rankdata[qid]:
                        qid_to_rankdata[qid][chunk_id] = score * DOCID_LENGTH

        query_idx += 1

    if batch_i % 10 == 0:
        print(f"  Batch {batch_i+1}/{len(eval_loader)}")

print(f"\nGenerated rankings for {len(qid_to_rankdata)} queries")

In [ ]:
# ============================================================
# 7B. Compute retrieval metrics
# ============================================================
# Build ground-truth qrels: each query has exactly 1 relevant chunk
qrel = {}
for qid, chunk_id in qid_to_chunk.items():
    qrel[qid] = {chunk_id: 1}  # binary relevance

# Compute metrics
hits_at_1 = 0
hits_at_5 = 0
hits_at_10 = 0
reciprocal_ranks = []

for qid in qid_to_rankdata:
    if qid not in qrel:
        continue
    relevant_docs = set(qrel[qid].keys())
    ranked = sorted(qid_to_rankdata[qid].items(), key=lambda x: x[1], reverse=True)
    ranked_docids = [d for d, _ in ranked]

    # Find rank of first relevant doc
    rr = 0.0
    for rank, chunk_id in enumerate(ranked_docids, 1):
        if chunk_id in relevant_docs:
            rr = 1.0 / rank
            if rank <= 1: hits_at_1 += 1
            if rank <= 5: hits_at_5 += 1
            if rank <= 10: hits_at_10 += 1
            break
    reciprocal_ranks.append(rr)

n_queries = len(reciprocal_ranks)
mrr = np.mean(reciprocal_ranks) if reciprocal_ranks else 0

print(f"{'='*60}")
print(f"  Retrieval Results ({n_queries} queries, {len(docid_to_tokenids)} chunks)")
print(f"{'='*60}")
print(f"  MRR:        {mrr:.4f}")
print(f"  Hits@1:     {hits_at_1}/{n_queries} ({100*hits_at_1/n_queries:.1f}%)")
print(f"  Hits@5:     {hits_at_5}/{n_queries} ({100*hits_at_5/n_queries:.1f}%)")
print(f"  Hits@10:    {hits_at_10}/{n_queries} ({100*hits_at_10/n_queries:.1f}%)")
print(f"{'='*60}")

# Break down by query version
print(f"\nBreakdown by query style:")
for ver in df['query_version'].unique():
    ver_queries = df[df['query_version'] == ver]
    ver_rrs = []
    ver_h1 = 0
    # Match by position in the dataset
    for idx, (_, row) in enumerate(ver_queries.iterrows()):
        qid = str(idx + (list(df['query_version'].unique()).index(ver)) * len(ver_queries))
        # Find the qid that matches this query text
        for q, qtext in qid_to_query.items():
            if qtext == row['query'] and q in qrel:
                ranked = sorted(qid_to_rankdata[q].items(), key=lambda x: x[1], reverse=True)
                relevant = set(qrel[q].keys())
                rr = 0.0
                for rank, d in enumerate([x[0] for x in ranked], 1):
                    if d in relevant:
                        rr = 1.0 / rank
                        if rank <= 1: ver_h1 += 1
                        break
                ver_rrs.append(rr)
                break
    if ver_rrs:
        print(f"  {ver:12s}: MRR={np.mean(ver_rrs):.4f}, Hits@1={ver_h1}/{len(ver_rrs)} ({100*ver_h1/len(ver_rrs):.1f}%)")

print(f"\nNOTE: This is memorization (train=test). High scores expected.")
print(f"The goal is to verify the pipeline works end-to-end.")

In [ ]:
# ============================================================
# 7C. Show detailed retrieval examples
# ============================================================
print("Sample Retrieval Results:")
print("=" * 90)

for qid in list(qid_to_rankdata.keys())[:15]:
    query = qid_to_query.get(qid, '?')
    gt_chunk = qid_to_chunk.get(qid, '?')
    ranked = sorted(qid_to_rankdata[qid].items(), key=lambda x: x[1], reverse=True)

    top1_chunk = ranked[0][0] if ranked else 'NONE'
    top1_score = ranked[0][1] if ranked else 0
    correct = 'OK' if top1_chunk == gt_chunk else 'X '

    # Get a snippet of the actual text
    gt_text = docs_df[docs_df['chunk_id'] == gt_chunk]['chunk_text'].values
    gt_snippet = gt_text[0][:60] + '...' if len(gt_text) > 0 else '?'

    print(f"  [{correct}] Q: {query[:70]}")
    print(f"       GT: {gt_chunk[:50]}...")
    print(f"       Top1: {top1_chunk[:50]}... (score={top1_score:.2f})")
    print(f"       Text: {gt_snippet}")
    print()

In [ ]:
# ============================================================
# 7D. Visualize retrieval performance
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MRR distribution
axes[0].hist(reciprocal_ranks, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(mrr, color='red', linestyle='--', label=f'Mean MRR={mrr:.3f}')
axes[0].set_title('Reciprocal Rank Distribution')
axes[0].set_xlabel('Reciprocal Rank')
axes[0].legend()

# Hits@k
k_values = [1, 5, 10]
hit_rates = [hits_at_1/n_queries, hits_at_5/n_queries, hits_at_10/n_queries]
axes[1].bar([f'@{k}' for k in k_values], hit_rates, color=['#4C72B0', '#55A868', '#DD8452'])
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Hits@K')
axes[1].set_ylabel('Hit Rate')
for i, (k, hr) in enumerate(zip(k_values, hit_rates)):
    axes[1].text(i, hr + 0.02, f'{hr:.1%}', ha='center', fontsize=12)

# Retrieved docs count per query
n_retrieved = [len(qid_to_rankdata[qid]) for qid in qid_to_rankdata]
axes[2].hist(n_retrieved, bins=range(0, max(n_retrieved)+2), color='coral', edgecolor='white')
axes[2].set_title('Number of Retrieved Docs per Query')
axes[2].set_xlabel('# Docs Retrieved')

plt.tight_layout()
plt.show()

---
## 8. Analyze Chrono-Semantic DocID Properties

In [ ]:
# ============================================================
# 8A. Verify chrono prefix structure
# ============================================================
print("Chrono-Semantic DocID Analysis:")
print("=" * 60)

year_dist = Counter()
month_dist = Counter()

for chunk_id, smtid in docid_to_smtid.items():
    year = YEAR_RANGE[0] + smtid[0]
    month = smtid[1] + 1
    year_dist[year] += 1
    month_dist[month] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Year distribution in DocIDs
years = sorted(year_dist.keys())
axes[0].bar(years, [year_dist[y] for y in years], color='steelblue')
axes[0].set_title('DocID Year Token Distribution')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')

# Month distribution
months = sorted(month_dist.keys())
axes[1].bar(months, [month_dist[m] for m in months], color='coral')
axes[1].set_title('DocID Month Token Distribution')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Count')
axes[1].set_xticks(range(1, 13))

plt.tight_layout()
plt.show()

print(f"\nYear distribution: {dict(sorted(year_dist.items()))}")
print(f"Month distribution: {dict(sorted(month_dist.items()))}")

# Check how many months are actually used
active_months = sum(1 for m in range(1, 13) if month_dist.get(m, 0) > 0)
print(f"\nActive months: {active_months}/12 (some months may have no data)")

In [ ]:
# ============================================================
# 8B. RQ code distribution per codebook
# ============================================================
fig, axes = plt.subplots(1, NUM_RQ_CODEBOOKS, figsize=(4*NUM_RQ_CODEBOOKS, 3))
if NUM_RQ_CODEBOOKS == 1:
    axes = [axes]

for cb in range(NUM_RQ_CODEBOOKS):
    codes = [smtid[2 + cb] for smtid in docid_to_smtid.values()]
    axes[cb].hist(codes, bins=range(RQ_CODEBOOK_SIZE + 1), color=f'C{cb}', edgecolor='white')
    axes[cb].set_title(f'Codebook {cb}')
    axes[cb].set_xlabel('Code')
    n_unique = len(set(codes))
    axes[cb].set_ylabel('Count')
    axes[cb].text(0.95, 0.95, f'{n_unique}/{RQ_CODEBOOK_SIZE}\nunique',
                  transform=axes[cb].transAxes, ha='right', va='top', fontsize=9)

plt.suptitle('RQ Code Distribution per Codebook', fontsize=13)
plt.tight_layout()
plt.show()

---
## 9. Interactive Query Test

In [ ]:
# ============================================================
# 9A. Query the trained model interactively
# ============================================================

def retrieve(query_text, top_k=5):
    """Retrieve documents for a query using constrained beam search."""
    model.eval()
    inputs = tokenizer(
        query_text, return_tensors='pt', truncation=True, max_length=MAX_QUERY_LEN
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            prefix_allowed_tokens_fn=prefixer_fn,
            max_new_tokens=DOCID_LENGTH,
            num_beams=NUM_BEAMS,
            num_return_sequences=min(NUM_BEAMS, len(docid_to_tokenids)),
            output_scores=True,
            return_dict_in_generate=True,
        )

    results = []
    seen = set()
    seqs = outputs.sequences[:, 1:]
    scores = outputs.sequences_scores

    for i in range(len(seqs)):
        token_ids = seqs[i].cpu().tolist()[:DOCID_LENGTH]
        smtid_key = '_'.join(str(t) for t in token_ids)
        score = scores[i].item()

        if smtid_key in smtid_to_docids:
            for chunk_id in smtid_to_docids[smtid_key]:
                if chunk_id not in seen:
                    seen.add(chunk_id)
                    doc_row = docs_df[docs_df['chunk_id'] == chunk_id].iloc[0]
                    tokens = [tokenizer.convert_ids_to_tokens(t) for t in token_ids]
                    results.append({
                        'chunk_id': chunk_id[:60],
                        'date': doc_row['date'],
                        'text_snippet': doc_row['chunk_text'][:120] + '...',
                        'score': score,
                        'docid_tokens': tokens,
                    })
    return results[:top_k]


# Test queries — mix of in-distribution keywords, topics, and questions
test_queries = [
    "industrial expo Nashik business",
    "Jason Day golf Quail Hollow quadruple bogey",
    "South-East Governors Nigeria quit notice safety",
    "Charlotte teens arrested stolen vehicle robbery",
    "What did the South-East Governors decide about gas blocks?",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    print("-" * 80)
    results = retrieve(q)
    if not results:
        print("  No results (all generated sequences were invalid)")
    for r in results[:3]:
        print(f"  [{r['date']}] {r['chunk_id']}")
        print(f"    Score: {r['score']:.3f} | DocID: {r['docid_tokens']}")
        print(f"    Text: {r['text_snippet']}")
        print()

---
## 10. Summary & Next Steps

In [ ]:
# ============================================================
# 10. Print summary
# ============================================================
print(f"""
{'='*60}
  DSI-CL PoC Summary
{'='*60}

Model:        {MODEL_NAME}
Total params: {total_params:,}
LoRA params:  {trainable_params:,} ({100*trainable_params/total_params:.2f}%)
LoRA targets: {lora_targets}
LoRA rank:    {LORA_R}

Dataset:      {len(docs_df)} chunks from {docs_df['doc_id'].nunique()} articles
              {len(train_dataset)} query-doc pairs (3 query styles per chunk)
Date range:   {docs_df['date'].min()} to {docs_df['date'].max()}
DocID format: {DOCID_LENGTH} tokens (2 chrono + {NUM_RQ_CODEBOOKS} RQ)
RQ codebook:  {NUM_RQ_CODEBOOKS} x {RQ_CODEBOOK_SIZE} centroids
Vocab:        {base_vocab_size:,} -> {len(tokenizer):,} (+{num_added} special)

Training:     {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, batch={BATCH_SIZE}
Final loss:   {loss_history[-1]:.4f}

Retrieval:
  MRR:     {mrr:.4f}
  Hits@1:  {hits_at_1}/{n_queries} ({100*hits_at_1/n_queries:.1f}%)
  Hits@5:  {hits_at_5}/{n_queries} ({100*hits_at_5/n_queries:.1f}%)
  Hits@10: {hits_at_10}/{n_queries} ({100*hits_at_10/n_queries:.1f}%)

{'='*60}
NEXT STEPS:
  1. Scale to full CC-News dataset (~708K articles) or BBC News (~160K)
  2. Use M=6 codebooks, nbits=8 (256 centroids) for better DocID resolution
  3. Train/test split for generalization eval
  4. Implement filtered retrieval with [FILTER:YYYY-MM]
  5. Run continual learning across monthly time periods
  6. Compare CL strategies (replay/merge/compose/null_space)
  7. Compute Positive Forgetting Rate (PFR) across periods
{'='*60}
""")